In [2]:
import sys
import pickle
import random
from typing import Optional, Union

from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import SinglesEnv
from poke_env.environment.env import _EnvPlayer
from poke_env.battle import AbstractBattle, Battle
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player, RandomPlayer



import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from typing import Any, Dict, Optional

from gymnasium.spaces import Box, Discrete, Space, MultiDiscrete
import torch
import torch.nn as nn
from torch import multiprocessing
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor


from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (Compose, DoubleToFloat, ObservationNorm, StepCounter,
                          TransformedEnv)


#Custom Env pytorch tutorial
from typing import Optional
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import step_mdp
#---------------------------
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

In [2]:
"""
cd "C:\Austin\Self_Projects\Pokemon_Sim\pokemon-showdown"
node pokemon-showdown start --no-security
"""

'\ncd "C:\\Austin\\Self_Projects\\Pokemon_Sim\\pokemon-showdown"\nnode pokemon-showdown start --no-security\n'

In [3]:
import copy

In [4]:
#Hyperparams 
format="gen9ou"
device="cpu"

In [5]:
team1 = """
Meruem (Kingambit) @ Leftovers  
Ability: Supreme Overlord  
Tera Type: Ghost  
EVs: 164 HP / 252 Atk / 92 Spe  
Adamant Nature  
- Sucker Punch  
- Iron Head  
- Kowtow Cleave  
- Swords Dance  

Moebius (Deoxys-Speed) @ Life Orb  
Ability: Pressure  
Tera Type: Psychic  
EVs: 28 HP / 252 SpA / 228 Spe  
Modest Nature  
IVs: 0 Atk  
- Nasty Plot  
- Focus Blast  
- Psycho Boost  
- Shadow Ball  

Anomaly (Great Tusk) @ Booster Energy  
Ability: Protosynthesis  
Tera Type: Steel  
EVs: 252 Atk / 4 Def / 252 Spe  
Jolly Nature  
- Headlong Rush  
- Ice Spinner  
- Rapid Spin  
- Close Combat  

Yoshi (Dragonite) @ Choice Band  
Ability: Multiscale  
Shiny: Yes  
Tera Type: Normal  
EVs: 16 HP / 252 Atk / 240 Spe  
Adamant Nature  
- Outrage  
- Extreme Speed  
- Ice Spinner  
- Fire Punch  

Anomаly (Iron Moth) @ Booster Energy  
Ability: Quark Drive  
Tera Type: Ground  
EVs: 124 Def / 132 SpA / 252 Spe  
Timid Nature  
- Fiery Dance  
- Sludge Wave  
- Tera Blast  
- Toxic Spikes  

Siren (Primarina) @ Assault Vest  
Ability: Torrent  
Tera Type: Poison  
EVs: 76 HP / 252 SpA / 180 Spe  
Modest Nature  
IVs: 0 Atk  
- Surf  
- Moonblast  
- Whirlpool  
- Psychic Noise  
"""

In [6]:
team2 = """
Torkoal @ Heat Rock  
Ability: Drought  
Tera Type: Ground  
EVs: 104 HP / 252 SpA / 152 SpD  
Quiet Nature  
- Eruption  
- Overheat  
- Earthquake  
- Stealth Rock  

Hatterene @ Air Balloon  
Ability: Magic Bounce  
Tera Type: Ghost  
EVs: 252 HP / 252 Def / 4 SpD  
Relaxed Nature  
IVs: 0 Atk / 0 Spe  
- Trick Room  
- Psychic Noise  
- Dazzling Gleam  
- Healing Wish  

Raging Bolt @ Life Orb  
Ability: Protosynthesis  
Tera Type: Ghost  
EVs: 36 Def / 252 SpA / 220 Spe  
Modest Nature  
IVs: 20 Atk  
- Thunderclap  
- Weather Ball  
- Dragon Pulse  
- Solar Beam  

Slither Wing @ Assault Vest  
Ability: Protosynthesis  
Tera Type: Fire  
EVs: 40 HP / 252 Atk / 216 Spe  
Adamant Nature  
- U-turn  
- First Impression  
- Earthquake  
- Temper Flare  

Venusaur @ Life Orb  
Ability: Chlorophyll  
Tera Type: Fire  
EVs: 4 Atk / 252 SpA / 252 Spe  
Naive Nature  
- Growth  
- Giga Drain  
- Weather Ball  
- Earthquake  

Walking Wake @ Wise Glasses  
Ability: Protosynthesis  
Tera Type: Fairy  
EVs: 12 HP / 244 SpA / 252 Spe  
Timid Nature  
- Hydro Steam  
- Weather Ball  
- Draco Meteor  
- Flip Turn  

"""

In [7]:
class SmogonEnv(SinglesEnv):

    def __init__(
        self,
        #SinglesEnv Variables
        account_configuration1: Optional[AccountConfiguration] = None,
        account_configuration2: Optional[AccountConfiguration] = None,
        battle_format: str = "gen8randombattle",
        start_timer_on_battle_start: bool = True,
        strict = True,
        fake = True,

        #Custom Input
        team1: Optional[Union[str, Teambuilder]] = None,
        team2: Optional[Union[str, Teambuilder]] = None,
    ):

        SinglesEnv.__init__(
            self,
            #DoublesEnv Variables
            account_configuration1=account_configuration1,
            account_configuration2=account_configuration2,
            battle_format=battle_format,
            start_timer_on_battle_start=start_timer_on_battle_start,
            strict=strict,
            fake=fake,
            log_level=25
        )
        self.agent1.teampreview = self.teampreview
        self.agent2.teampreview = self.teampreview

        self.agent1.update_team(team1)
        self.agent2.update_team(team2)

    #SinglesEnv Required Functions:

    def calc_reward(self, battle) -> float:
        #Initially using built in reward helper function provided by Poke-Env
        #For simplicity of initiall implementation and testing

        return self.reward_computing_helper(
            battle, fainted_value=2.0, hp_value=1.0, victory_value=30.0
        )


    def embed_battle(self, battle: Battle): #Will write a custom reward function as required - __NEED TO UPDATE FOR DOUBLE BATTLE__
        assert isinstance(battle, Battle)
        # -1 indicates that the move does not have a base power
        # or is not available
        moves_base_power = -np.ones(4)
        moves_dmg_multiplier = np.ones(4)
        for i, move in enumerate(battle.available_moves):
            moves_base_power[i] = (
                move.base_power / 100
            )  # Simple rescaling to facilitate learning
            if battle.opponent_active_pokemon is not None:
                moves_dmg_multiplier[i] = move.type.damage_multiplier(
                    battle.opponent_active_pokemon.type_1,
                    battle.opponent_active_pokemon.type_2,
                    type_chart=battle.opponent_active_pokemon._data.type_chart,
                )

        # We count how many pokemons have fainted in each team
        fainted_mon_team = len([mon for mon in battle.team.values() if mon.fainted]) / 6
        fainted_mon_opponent = (
            len([mon for mon in battle.opponent_team.values() if mon.fainted]) / 6
        )

        # Final vector with 10 components
        final_vector = np.concatenate(
            [
                moves_base_power,
                moves_dmg_multiplier,
                [fainted_mon_team, fainted_mon_opponent],
            ]
        )
        return torch.Tensor(final_vector)

        
    
    def teampreview(self, battle: Battle) -> str: #Will write a custom reward function as required
        members = [1,2,3,4,5,6]#list(range(1, 7))
        random.shuffle(members)
        team_string = "/team " + "".join([str(x) for x in members])
        print(team_string)
        return team_string

    #Helper Functions:

    def print_teams(self):
        print(self.agent1._team.yield_team())
        print(self.agent2._team.yield_team())

    def print_torchrl_env_stats(self):
        print(self.device)
        print(self.batch_size)

    def select_game_team(self):
        pass

    def set_battle(self):
        self.battle1._finished=False
        self.battle2._finished=False

In [8]:
env = SmogonEnv(battle_format=format, strict=True, fake = False, team1=team1, team2=team2, start_timer_on_battle_start=False)

/team 135642
/team 625134


In [9]:
env.reset()

({'SmogonEnv 8u6ja': tensor([0.7000, 0.8000, 0.8500, 0.0000, 1.0000, 0.5000, 1.0000, 1.0000, 0.0000,
          0.0000]),
  'SmogonEnv wp1zb': tensor([0.8000, 0.5000, 1.3000, 0.6000, 1.0000, 0.5000, 0.5000, 1.0000, 0.0000,
          0.0000])},
 {'SmogonEnv 8u6ja': {}, 'SmogonEnv wp1zb': {}})

In [28]:
td = {
        "actions":{
                env.agents[0]: (
                    torch.Tensor([-2]).type(torch.int64)
                ),

                env.agents[1]: (
                    torch.Tensor([-1]).type(torch.int64)
                )
            }}
env.step(td["actions"])

({'SmogonEnv 8u6ja': tensor([0.0000, 1.2000, 1.4000, 0.8000, 1.0000, 0.2500, 0.5000, 2.0000, 0.1667,
          0.3333]),
  'SmogonEnv wp1zb': tensor([0.0000, 0.7500, 0.8000, 0.0000, 0.5000, 0.5000, 1.0000, 0.5000, 0.3333,
          0.1667])},
 {'SmogonEnv 8u6ja': 30.0, 'SmogonEnv wp1zb': -30.000000000000004},
 {'SmogonEnv 8u6ja': False, 'SmogonEnv wp1zb': False},
 {'SmogonEnv 8u6ja': True, 'SmogonEnv wp1zb': True},
 {'SmogonEnv 8u6ja': {}, 'SmogonEnv wp1zb': {}})

In [11]:
bat_read_reg_state1 = copy.deepcopy(env.battle1)
bat_read_reg_state2 = copy.deepcopy(env.battle2)

In [13]:
mon_dead_state1 = copy.deepcopy(env.battle1)
mon_dead_state2 = copy.deepcopy(env.battle2)

In [47]:
bat_read_sun_state1 = copy.deepcopy(env.battle1)
bat_read_sun_state2 = copy.deepcopy(env.battle2)

In [17]:
bat_read_pre_chband_state1 = copy.deepcopy(env.battle1)
bat_read_pre_chband_state2 = copy.deepcopy(env.battle2)

In [19]:
bat_read_chband_state1 = copy.deepcopy(env.battle1)
bat_read_chband_state2 = copy.deepcopy(env.battle2)

In [23]:
#Chlorophyl
bat_read_abi_act_state1 = copy.deepcopy(env.battle1)
bat_read_abi_act_state2 = copy.deepcopy(env.battle2)

In [26]:
bat_read_tr_act_state1 = copy.deepcopy(env.battle1)
bat_read_tr_act_state2 = copy.deepcopy(env.battle2)

In [29]:
bat_read_forfit_state1 = copy.deepcopy(env.battle1)
bat_read_forfit_state2 = copy.deepcopy(env.battle2)

In [47]:
all_bat_env_states ={
    "mon_dead_waiting_to_switch":[mon_dead_state1,mon_dead_state2],
    "battle_ready_regular_state":[bat_read_reg_state1,bat_read_reg_state2],
    #"battle_ready_sun_state":[bat_read_sun_state1,bat_read_sun_state2],
    "battle_ready_pre_choice_band_dnite_state":[bat_read_pre_chband_state1,bat_read_pre_chband_state2],
    "battle_ready_post_choice_band_dnite_state":[bat_read_chband_state1,bat_read_chband_state2],
    "battle_ready_ability_active_chlorophill_vena_state":[bat_read_abi_act_state1,bat_read_abi_act_state2],
    "battle_ready_TR_active_haterenne_state":[bat_read_tr_act_state1,bat_read_tr_act_state2],
    "battle_post_forfeit_state":[bat_read_forfit_state1,bat_read_forfit_state2],
}


In [ ]:
current_game_save_path = "C:\Austin\Self_Projects\Pokemon_Sim\Reg_J_simulation_project\Singles_env_testing\env_stat_test_games\simu_game_test_1\Gen9OU-2025-11-12-smogonenv8u6ja-smogonenvwp1zb.pickle"
with open(current_game_save_path, 'wb') as handle:
    pickle.dump(all_bat_env_states, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [44]:
espeed.entry

{'accuracy': 100,
 'basePower': 80,
 'category': 'Physical',
 'contestType': 'Cool',
 'flags': {'contact': 1, 'metronome': 1, 'mirror': 1, 'protect': 1},
 'name': 'Extreme Speed',
 'num': 245,
 'pp': 5,
 'priority': 2,
 'secondary': None,
 'target': 'normal',
 'type': 'Normal'}

In [42]:
bat_read_chband_state1

In [39]:
bat_read_pre_chband_state1.available_moves

[outrage (Move object),
 extremespeed (Move object),
 icespinner (Move object),
 firepunch (Move object)]

In [30]:
mon_dead_state1.available_switches

[greattusk (pokemon object) [Active: False, Status: None],
 ironmoth (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None],
 deoxysspeed (pokemon object) [Active: False, Status: None]]

In [35]:
mon_dead_state2.all_active_pokemons

[walkingwake (pokemon object) [Active: True, Status: FNT],
 kingambit (pokemon object) [Active: True, Status: None]]

In [32]:
bat_read_reg_state1.available_switches


[greattusk (pokemon object) [Active: False, Status: None],
 ironmoth (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None],
 deoxysspeed (pokemon object) [Active: False, Status: None]]

In [34]:
bat_read_reg_state2.all_active_pokemons

[walkingwake (pokemon object) [Active: True, Status: None],
 kingambit (pokemon object) [Active: True, Status: None]]

In [37]:
bat_read_forfit_state1.available_switches

[greattusk (pokemon object) [Active: False, Status: None],
 ironmoth (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None]]